# 06 · 评测与反馈

评测策略、诊断失败、回流恢复数据，并验证模型更新效果。

In [ ]:
import json
import uuid

from examples._display import (
    setup_notebook,
    show_animation,
    show_image,
    show_metrics,
    show_pipeline,
)

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("evaluation_and_feedback")

from physrealx import ArtifactRef, PhysRealX, RunSpec, VersionedRef
from physrealx.domain import TrainingSpec
from physrealx.integrations.act import ACTTrainer
from physrealx.integrations.datasets import dataset_bundle
from physrealx.integrations.feedback import (
    DeploymentEvidence,
    FailureQuery,
    PolicyInterventionEvidence,
    episode_metrics,
)
from physrealx.runtime import load_episode

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")

## 1. 输入

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/data_management_and_training/output/results.json").read_text()
)
model_ref = VersionedRef.model_validate(upstream["model"])
evaluation_run = RunSpec.model_validate(upstream["evaluation_run"])
snapshot_ref = ArtifactRef.model_validate(upstream["snapshot"])
run_key = model_ref.artifact_id.removeprefix("art_")[:12]

show_pipeline(
    {
        "策略评测": "独立闭环 Episode",
        "轨迹诊断": "感知、规划、控制与接触",
        "定向恢复": "同场景、任务和种子",
        "策略更新": "恢复示范增广训练",
        "闭环验证": "独立种子与配对回放",
    }
)

## 2. 基线评测与日志归档

In [ ]:
evaluation_ref = px.evaluations.submit(
    resource_id=f"laptop1.act.feedback.{run_key}",
    version="1.0.0",
    model=model_ref,
    run=evaluation_run,
    episode_count=3,
    base_seed=101000,
)
evaluation = px.evaluations.get_report(evaluation_ref)
evidence = px.artifacts.put_json(DeploymentEvidence(dataset=evaluation.dataset))
archive = px.telemetry.record(evidence, method="episode_telemetry")
telemetry_ref = archive.output("telemetry")
telemetry_manifest_ref = archive.output("telemetry_manifest")
telemetry_manifest = json.loads(px.artifacts.get_bytes(telemetry_manifest_ref))
show_metrics(
    {
        "基线成功率": evaluation.success_rate,
        "Episode": int(evaluation.metrics["episode_count"]),
        "数据来源": next(iter(telemetry_manifest["source_counts"])),
        "真机证据": telemetry_manifest["live_robot_execution"],
    }
)

## 3. 失败诊断与校准决策

In [ ]:
diagnosis = px.failures.analyze(telemetry_ref, method="trajectory_diagnostics")
failure_report_ref = diagnosis.output("failure_report")
failure_report = json.loads(px.artifacts.get_bytes(failure_report_ref))
calibration_plan_ref = px.feedback.plan_physics_calibration(telemetry_ref, failure_report_ref)
calibration_plan = json.loads(px.artifacts.get_bytes(calibration_plan_ref))
show_metrics(
    {
        "失败 Episode": len(failure_report["failures"]),
        "症状类别": ", ".join(sorted({row["category"] for row in failure_report["failures"]})),
        "物理校准": calibration_plan["status"],
    }
)

## 4. 失败回流与定向恢复

In [ ]:
if not failure_report["failures"]:
    raise RuntimeError("基线评测没有失败样本，无法演示反馈闭环")
query = px.artifacts.put_json(
    FailureQuery(
        report=failure_report_ref,
        resource_id=f"laptop1.feedback.{run_key}",
    )
)
selection = px.feedback.build_dataset(query, method="failure_episodes")
selected_id = f"laptop1.feedback.{run_key}"
selected_ref = next(
    (
        ref
        for ref in px.datasets.list()
        if ref.resource_id == selected_id and ref.version == "1.0.0"
    ),
    None,
)
if selected_ref is None:
    selected_dataset = dataset_bundle(px.artifacts, selection.output("dataset"))
    selected_ref = px.datasets.register(selected_dataset)
else:
    selected_dataset = px.datasets.get(selected_ref)
selected = selected_dataset.episodes

recovery_refs, lineage = [], []
for index, summary in enumerate(selected):
    failed = load_episode(px.artifacts, summary)
    recovery_id = f"laptop1.recovery.{run_key}.{index}"
    recovered_ref = next(
        (
            ref
            for ref in px.datasets.list()
            if ref.resource_id == recovery_id and ref.version == "1.0.0"
        ),
        None,
    )
    if recovered_ref is None:
        recovery_run = RunSpec(
            run_id=str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    f"physrealx:feedback-recovery:{run_key}:{failed.seed}",
                )
            ),
            scene=failed.scene,
            task=failed.task,
            backend=failed.backend,
            policy=evaluation_run.policy,
            episode_count=1,
            base_seed=failed.seed,
            stop_on_success=False,
            dataset_id=recovery_id,
            dataset_version="1.0.0",
            record_signals=evaluation_run.record_signals,
            tags=("expert_demonstration", "failure_recovery", failed.episode_id),
        )
        result = px.runs.submit(recovery_run)
        if result.result is None or result.result.dataset is None:
            raise RuntimeError("定向恢复未产生数据集")
        recovered_ref = result.result.dataset
    recovered = px.datasets.get(recovered_ref)
    if not all(row.success for row in recovered.episodes):
        raise RuntimeError("专家恢复未成功")
    recovery_refs.append(recovered_ref)
    lineage.append(
        {
            "failed_episode": failed.episode_id,
            "recovery_dataset": recovered_ref.model_dump(mode="json"),
            "seed": failed.seed,
        }
    )

intervention_ref = px.artifacts.put_json(
    PolicyInterventionEvidence(
        failure_report=failure_report_ref,
        recovery_datasets=tuple(recovery_refs),
    )
)
causal_report_ref = px.feedback.analyze_policy_intervention(intervention_ref)
causal_report = json.loads(px.artifacts.get_bytes(causal_report_ref))
show_metrics(
    {
        "失败样本": len(selected),
        "恢复示范": len(recovery_refs),
        "配对因果验证": causal_report["validated_count"],
    }
)

## 5. 策略更新

In [ ]:
augmented_ref = px.datasets.training_subset(
    snapshot_ref,
    resource_id=f"laptop1.feedback.train.{run_key}",
    augmentations=tuple(recovery_refs),
    required_tag="expert_demonstration",
)
initial_model = px.training.get_model(model_ref)
config = dict(initial_model.training_config["configuration"])
config.update(
    steps=max(600, int(config["steps"])),
    learning_rate=min(1e-4, float(config["learning_rate"])),
)
updated_model_ref = px.training.submit(
    TrainingSpec(
        resource_id=f"laptop1.act.updated.{run_key}",
        version="1.0.0",
        dataset=augmented_ref,
        trainer=ACTTrainer.descriptor.bind(
            checkpoint=initial_model.weights,
            config=config,
        ),
    )
)
updated_model = px.training.get_model(updated_model_ref)
if updated_model.weights == initial_model.weights:
    raise RuntimeError("再训练没有产生新权重")
show_metrics({"更新模型": updated_model_ref.resource_id, **updated_model.metrics})

## 6. 独立对照评测

In [ ]:
reports, references, contacts = {}, {}, {}
for name, candidate in (("initial", model_ref), ("updated", updated_model_ref)):
    reference = px.evaluations.submit(
        resource_id=f"laptop1.act.heldout.{name}.{run_key}",
        version="1.0.0",
        model=candidate,
        run=evaluation_run,
        episode_count=3,
        base_seed=111000,
    )
    report = px.evaluations.get_report(reference)
    reports[name], references[name], contacts[name] = report, reference, []
    for summary in px.datasets.get(report.dataset).episodes:
        episode = load_episode(px.artifacts, summary)
        task = px.tasks.get(episode.task)
        contacts[name].append(episode_metrics(px.artifacts, episode, task))
policy_update_ref = px.feedback.assess_policy_update(references["initial"], references["updated"])
policy_update = json.loads(px.artifacts.get_bytes(policy_update_ref))
show_metrics(
    {
        "初始成功率": reports["initial"].success_rate,
        "更新成功率": reports["updated"].success_rate,
        "策略门控": policy_update["status"],
        "更新峰值接触力均值 (N)": sum(row["peak_contact_force_n"] for row in contacts["updated"])
        / len(contacts["updated"]),
    }
)

## 7. 反馈闭环可视化

In [ ]:
failed_dataset = px.datasets.get(evaluation.dataset)
failed_id = failure_report["failures"][0]["episode_id"]
failed_summary = next(row for row in failed_dataset.episodes if row.episode_id == failed_id)
failed_episode = load_episode(px.artifacts, failed_summary)
recovery_dataset = px.datasets.get(recovery_refs[0])
recovery_summary = recovery_dataset.episodes[0]

replay_ref = px.evaluations.submit(
    resource_id=f"laptop1.act.replay.updated.{run_key}",
    version="1.0.0",
    model=updated_model_ref,
    run=evaluation_run,
    episode_count=1,
    base_seed=failed_episode.seed,
)
replay_report = px.evaluations.get_report(replay_ref)
replay_summary = px.datasets.get(replay_report.dataset).episodes[0]

dashboard_ref = px.feedback.preview_report(
    references["initial"],
    references["updated"],
    failure_report_ref,
    causal_report_ref,
    calibration_plan_ref,
    policy_update_ref,
)
cycle_ref = px.feedback.preview_cycle(
    failed_summary.artifact,
    recovery_summary.artifact,
    replay_summary.artifact,
    frames=8,
    fps=3,
)
show_image(px.artifacts.get_bytes(dashboard_ref))
show_animation(px.artifacts.get_bytes(cycle_ref))

## 8. 输出

In [ ]:
lineage_ref = px.artifacts.put_json(
    {
        "schema": "physrealx.recovery-lineage/3",
        "links": lineage,
        "initial_model": model_ref.model_dump(mode="json"),
        "updated_model": updated_model_ref.model_dump(mode="json"),
        "training_dataset": augmented_ref.model_dump(mode="json"),
        "paired_success_rates": {key: value.success_rate for key, value in reports.items()},
        "policy_update_status": policy_update["status"],
    }
)
results = {
    "initial_model": model_ref.model_dump(mode="json"),
    "model": updated_model_ref.model_dump(mode="json"),
    "evaluations": {name: ref.model_dump(mode="json") for name, ref in references.items()},
    "failure_report": failure_report_ref.model_dump(mode="json"),
    "causal_failure_report": causal_report_ref.model_dump(mode="json"),
    "physics_calibration_plan": calibration_plan_ref.model_dump(mode="json"),
    "policy_update_assessment": policy_update_ref.model_dump(mode="json"),
    "feedback_dataset": selected_ref.model_dump(mode="json"),
    "training_dataset": augmented_ref.model_dump(mode="json"),
    "recovery_datasets": [ref.model_dump(mode="json") for ref in recovery_refs],
    "feedback_dashboard": dashboard_ref.model_dump(mode="json"),
    "feedback_cycle": cycle_ref.model_dump(mode="json"),
}
px.artifacts.export(
    OUTPUT_DIR,
    results,
    files={
        "failure-report.json": failure_report_ref,
        "causal-failure-report.json": causal_report_ref,
        "physics-calibration-plan.json": calibration_plan_ref,
        "policy-update-assessment.json": policy_update_ref,
        "recovery-lineage.json": lineage_ref,
        "feedback-dashboard.png": dashboard_ref,
        "feedback-cycle.gif": cycle_ref,
    },
)
print("结果索引:", OUTPUT_DIR / "results.json")